# Working with DLIS Files

DLIS (Digital Log Interchange Standard) is a binary well log format commonly used in the oil & gas industry. Unlike LAS files, DLIS files can contain:

- Multiple logical files per physical file
- Multiple frames (data tables) per logical file
- Multi-dimensional curve data
- Rich metadata about the well and logging operation

Welly supports reading DLIS files through the `Well.from_dlis()` method.

## Installation

DLIS support requires the `dlisio` library. Install it with:

```bash
pip install welly[dlis]
```

Or install dlisio separately:

```bash
pip install dlisio
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import welly
from welly import Well

print(f"welly version: {welly.__version__}")

## Loading a DLIS File

We'll use an Ocean Drilling Program (ODP) DLIS file as an example. This file contains logging data from Site 1218A, a scientific drilling site in the Pacific Ocean.

In [ ]:
# Load the DLIS file
well = Well.from_dlis('data/199-1218A_std-proc.dlis')

print(f"Well Name: {well.name}")
print(f"Number of curves: {len(well.data)}")

In [ ]:
# List all available curves
print("Available curves:")
for name in sorted(well.data.keys()):
    curve = well.data[name]
    print(f"  {name:8}: {curve.units:10} ({len(curve)} samples)")

## Exploring the Data

Let's look at the depth range and some key curves.

In [ ]:
# Get depth information
first_curve = list(well.data.values())[0]
depth = first_curve.df.index

print(f"Depth range: {depth.min():.1f} to {depth.max():.1f} m")
print(f"Number of samples: {len(depth)}")
print(f"Sample interval: {np.median(np.diff(depth)):.2f} m")

In [ ]:
# Look at key petrophysical curves
key_curves = ['HCGR', 'RHOB', 'ILD', 'APLC', 'SP']

print("Key curve statistics:")
print("-" * 50)
for name in key_curves:
    if name in well.data:
        curve = well.data[name]
        values = curve.values
        valid = values[~np.isnan(values)]
        print(f"{name:8}: {valid.min():10.3f} - {valid.max():10.3f} {curve.units}")

## Plotting DLIS Data

Once loaded, DLIS wells work exactly like LAS wells.

In [ ]:
# Create a multi-track plot
fig, axes = plt.subplots(1, 4, figsize=(14, 10), sharey=True)

# Gamma Ray
ax = axes[0]
ax.plot(well.data['HCGR'].values, depth, 'g-', lw=0.5)
ax.set_xlabel('GR (gAPI)')
ax.set_ylabel('Depth (m)')
ax.set_xlim(0, 25)
ax.set_title('Gamma Ray')

# Density
ax = axes[1]
ax.plot(well.data['RHOB'].values, depth, 'r-', lw=0.5)
ax.set_xlabel('RHOB (g/cm3)')
ax.set_xlim(1.0, 2.5)
ax.set_title('Density')

# Resistivity
ax = axes[2]
ax.plot(well.data['ILD'].values, depth, 'k-', lw=0.5)
ax.set_xlabel('ILD (ohm.m)')
ax.set_xscale('log')
ax.set_xlim(0.1, 1000)
ax.set_title('Resistivity')

# Caliper
ax = axes[3]
ax.plot(well.data['CALI'].values, depth, 'b-', lw=0.5)
ax.set_xlabel('CALI (in)')
ax.set_xlim(0, 20)
ax.set_title('Caliper')

for ax in axes:
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3)

plt.suptitle(f'ODP Site {well.name} - DLIS Data', fontsize=14)
plt.tight_layout()
plt.show()

## About This Data

This is data from **Ocean Drilling Program (ODP) Site 1218A**, a scientific drilling site in the equatorial Pacific Ocean. The data characteristics are quite different from typical oil & gas wells:

- **Very low gamma ray** (0-20 gAPI) - typical of marine carbonates and siliceous ooze
- **Low density** (~1.5-2.0 g/cm3) - unconsolidated marine sediments
- **Depth below seafloor** - the depth index starts at -74m (above mudline) and goes to ~915m below seafloor
- **No hydrocarbons** - this is a scientific well, not an exploration well

This makes it an interesting example for demonstrating DLIS loading, but the petrophysical interpretation would be quite different from a conventional reservoir.

## DLIS Metadata

DLIS wells store additional metadata about the source file:

In [ ]:
# Access DLIS-specific metadata
if hasattr(well, '_dlis_frame'):
    print(f"Frame: {well._dlis_frame}")
if hasattr(well, '_dlis_index_type'):
    print(f"Index type: {well._dlis_index_type}")

## Exporting to LAS

You can export DLIS data to LAS format for use with other software:

In [ ]:
# Export to LAS (uncomment to run)
# well.to_las('data/1218A_from_dlis.las')
# print("Exported to LAS format")

## Loading Specific Frames

DLIS files can contain multiple frames with different sampling rates or curve sets. Use `describe_dlis()` to find the frame you need, then load it by name:

In [ ]:
# Load a specific frame by name
# well = Well.from_dlis('file.dlis', frame='60B')

# Load from a specific logical file
# well = Well.from_dlis('file.dlis', logical_file=1)

# Load all wells from the file
# wells = Well.from_dlis('file.dlis', return_all=True)

## Error Handling

DLIS files can sometimes be malformed. The `error_handling` parameter controls how errors are handled:

- `'warn'` (default): Log warnings but continue parsing
- `'strict'`: Raise exceptions on errors
- `'ignore'`: Silently ignore errors

In [ ]:
# Lenient error handling for problematic files
# well = Well.from_dlis('file.dlis', error_handling='ignore')

## Sample DLIS Files

If you need more DLIS files for testing:

- **Utah FORGE project**: https://gdr.openei.org/submissions/1330 (FMI images, CBL data, standard logs)
- **IODP/ODP data**: https://web.iodp.tamu.edu/LORE/ (scientific drilling data)

## Summary

Key points for working with DLIS files:

1. Install with `pip install welly[dlis]`
2. Use `Well.from_dlis()` to load curve data
3. Specify `frame='name'` to load a specific frame
4. Use `return_all=True` to get all wells from a file
5. Once loaded, DLIS wells work like any other welly Well
6. Export to LAS with `well.to_las()` if needed